In [1]:
# Ensure project root on sys.path for `src` imports
import sys
from pathlib import Path

cwd = Path.cwd()
for base in [cwd, cwd.parent, cwd.parent.parent]:
    if (base / "src").exists():
        sys.path.insert(0, str(base))
        break


# Student Guide: Initialization

Goal: fetch Shakespeare texts into `data/raw/` to start the pipeline.

What you’ll learn:
- Using `requests` to download files over HTTP.
- Handling basic errors and rate limiting.
- Respecting a data source’s terms of use.

Workflow:
1) Choose Gutenberg IDs (e.g., 100 = Complete Works).
2) Try several common URL patterns.
3) Save the text under `data/raw/`.

References:
- Requests (HTTP): https://requests.readthedocs.io/en/latest/
- Project Gutenberg Terms: https://www.gutenberg.org/policy/terms_of_use.html
- Good-citizen crawling tips: https://developers.google.com/search/docs/crawling-indexing/robots/intro

Try this:
- Add more Gutenberg IDs; compare file sizes and content.
- Add simple caching or backoff if you frequently re-run downloads.


# 00 - Initialize Raw Datasets (Project Gutenberg)

This notebook downloads Shakespeare texts from Project Gutenberg into `data/raw/`.
- Default: the complete works (PG ID 100).
- You can add more Gutenberg IDs to the list below.

Note: Please follow Project Gutenberg's Terms of Use and rate limits.


In [2]:
from pathlib import Path
from typing import List
import time

import requests

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Add more Gutenberg IDs as needed
GUTENBERG_IDS: List[int] = [100]  # 100 = The Complete Works of William Shakespeare

# Candidate URL patterns to try per ID (most common first)
URL_PATTERNS = [
    "https://www.gutenberg.org/files/{id}/{id}-0.txt",
    "https://www.gutenberg.org/files/{id}/{id}.txt",
    "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt",
    "https://www.gutenberg.org/ebooks/{id}.txt.utf-8",
]

HEADERS = {"User-Agent": "shakespeare-rag/0.1 (educational; contact: n/a)"}


def fetch_gutenberg_text(gid: int, out_dir: Path = RAW_DIR, sleep_s: float = 1.0) -> Path:
    out_path = out_dir / f"gutenberg_{gid}.txt"
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    last_error = None
    for pattern in URL_PATTERNS:
        url = pattern.format(id=gid)
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200 and r.text.strip():
                out_path.write_text(r.text, encoding="utf-8")
                time.sleep(sleep_s)
                return out_path
            last_error = f"status={r.status_code}"
        except Exception as e:
            last_error = str(e)
    raise RuntimeError(f"Failed to fetch Gutenberg ID {gid}: {last_error}")


downloaded = []
for gid in GUTENBERG_IDS:
    try:
        path = fetch_gutenberg_text(gid)
        downloaded.append(path)
        print(f"Downloaded {gid} -> {path}")
    except Exception as e:
        print(f"ERROR downloading {gid}: {e}")

print("\nSummary:")
for p in downloaded:
    print("-", p, f"({p.stat().st_size} bytes)")



Downloaded 100 -> data/raw/gutenberg_100.txt

Summary:
- data/raw/gutenberg_100.txt (5422721 bytes)
